# Ноутбук 03 — Новые признаки и обработка данных

Это самый важный технический ноутбук. Здесь две темы:

1. **Feature engineering** — создание новых признаков из старых. Пример: вместо отдельных «криминал» и «нет страховки» даём модели готовое «криминал × нет страховки» — так ей легче учиться.
2. **Препроцессинг** — подготовка чисел для модели: заполнить пропуски (медианой) и привести к одному масштабу (стандартизация).

Весь этот код живёт в `model_utils.py` — ноутбук показывает его в действии на живых примерах.

In [ ]:
# Переходим в корень проекта (если надо) и импортируем всё нужное.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd  # таблицы
import numpy as np  # математика
# Импортируем НАШИ функции из model_utils.py — те самые, что используют train.py и predict.py.
from model_utils import (
    generate_features,  # создание 9 новых признаков
    get_numeric_feature_names,  # отбор числовых признаков
    numeric_matrix,  # DataFrame -> матрица numpy
    fit_preprocessor,  # выучить медианы/средние/масштабы на train
    transform_features,  # применить их (к train или test)
    fit_categorical_modes,  # выучить моды текстовых колонок
    impute_categorical,  # заполнить текстовые пропуски модами
    CATEGORICAL_COLUMNS,  # список текстовых колонок
)
# Читаем данные.
train = pd.read_csv("hard_train.csv")
test = pd.read_csv("hard_test.csv")
print("train:", train.shape, " test:", test.shape)

## 1. Создаём новые признаки одной функцией

In [ ]:
# Применяем generate_features: на вход таблица, на выход — НОВАЯ таблица с +9 колонками.
# Старая таблица train при этом НЕ меняется (функция работает с копией).
enriched = generate_features(train)
print("Было колонок:", train.shape[1], " стало:", enriched.shape[1])
# Какие именно колонки добавились? Вычитаем множества имён.
new_columns = [c for c in enriched.columns if c not in train.columns]
print("Новые признаки:", new_columns)

## 2. Разбираем каждый новый признак на живых примерах

Возьмём трёх клиентов и посмотрим, как из их исходных данных получаются новые числа.

In [ ]:
# Берём 3 первых клиентов и показываем исходные колонки, из которых строятся property_exposure и cyber_exposure.
cols = ["crime_rate", "property_insurance", "cyber_risk", "cyber_protection"]
print(enriched[cols].head(3).round(2))
print()
# А вот что получилось: exposure = риск * (1 - страховка). Проверяем руками первую строку:
# crime_rate=41.81, property_insurance=0 -> 41.81 * (1-0) = 41.81 (риск ничем не покрыт!).
print(enriched[["property_exposure", "cyber_exposure"]].head(3).round(2))

In [ ]:
# Индекс цифровой уязвимости: активность * число ОТСУТСТВУЮЩИХ защит (их всего 3).
# Клиент 0: score=39.12, защиты 0+1+1=2 -> отсутствует 1 -> 39.12 * 1 = 39.12.
cols = ["digital_behavior_score", "password_manager", "two_factor_auth", "security_training", "cyber_vulnerability"]
print(enriched[cols].head(3).round(2))

In [ ]:
# Финансовые отношения и остальное: смотрим исходники и результат рядом.
cols = ["income", "average_balance", "loan_amount", "income_to_balance_ratio", "credit_load_ratio"]
print(enriched[cols].head(3).round(4))
print()
cols2 = ["website_visits", "mobile_sessions", "digital_channel_ratio", "active_policies", "expired_policies", "net_active_policies"]
print(enriched[cols2].head(3).round(3))
print()
# Возраст: квадрат и флаг молодости (1, если младше 25).
print(enriched[["age", "age_squared", "is_young"]].head(5))

Зачем всё это? Модели-деревья (CatBoost) любят готовые комбинации: «высокий риск И нет защиты» одним числом учить проще, чем выводить из двух. Эти признаки реально вошли в топ важных (см. ноутбук 04).

## 3. Шаг 1 препроцессинга: пропуски заполняем медианой

**Медиана** — среднее по порядку значение. Пример: в ряду [1, 2, 100] среднее = 34.3 (испорчено выбросом 100), а медиана = 2 (устойчива). Поэтому пропуски заполняют медианой, а не средним.

In [ ]:
# Отбираем числовые признаки (всё числовое, кроме customer_id и protection_score).
feature_names = get_numeric_feature_names(enriched)
print("Числовых признаков:", len(feature_names))
# Превращаем в матрицу numpy: строки — клиенты, столбцы — признаки.
raw_matrix = numeric_matrix(enriched, feature_names)
print("Размер матрицы:", raw_matrix.shape)
# Считаем пропуски (NaN) в матрице: np.isnan — True где NaN; .sum() — сколько всего.
print("Всего пропусков в матрице:", int(np.isnan(raw_matrix).sum()))
# В каких колонках? Считаем NaN по столбцам (axis=0) и показываем ненулевые.
nan_per_column = np.isnan(raw_matrix).sum(axis=0)
for name, count in zip(feature_names, nan_per_column):
    if count > 0:
        print(f"  {name}: {count} пропусков")

In [ ]:
# Учим препроцессор: считаем медианы (а также средние и масштабы — они понадобятся дальше).
# ВАЖНО: fit_preprocessor вызывается на TRAIN — параметры запоминают свойства обучающих данных.
medians, means, scales = fit_preprocessor(raw_matrix)
# Посмотрим медианы трёх колонок с пропусками: найдём их позиции в списке признаков.
for name in ["house_value", "car_value", "average_claim_cost"]:
    position = feature_names.index(name)  # позиция колонки в матрице
    print(f"Медиана {name}: {medians[position]:.2f}")
# Мини-пример медианы руками: ряд [1, 2, 100] -> медиана 2, среднее 34.33.
print("Пример: медиана [1, 2, 100] =", np.median([1, 2, 100]), ", среднее =", round(np.mean([1, 2, 100]), 2))

## 4. Шаг 2 препроцессинга: стандартизация

Признаки живут в разных масштабах: доход — тысячи, флаги — 0/1. **Стандартизация** приводит всех к одному масштабу: `(x - среднее) / разброс`. После неё у каждой колонки среднее ≈ 0 и разброс ≈ 1.

In [ ]:
# Применяем препроцессор: заполнить пропуски медианами + стандартизировать.
transformed = transform_features(raw_matrix, medians, means, scales)
print("Размер после обработки:", transformed.shape)
# Проверка 1: пропусков не осталось (должно быть 0).
print("Пропусков после обработки:", int(np.isnan(transformed).sum()))
# Проверка 2: среднее каждой колонки стало ~0, разброс ~1. Покажем первые 5 колонок.
print("Средние первых 5 колонок:", transformed[:, :5].mean(axis=0).round(6))
print("Разбросы первых 5 колонок:", transformed[:, :5].std(axis=0).round(6))

## 5. Главное правило: fit — на train, transform — везде

Параметры (медианы/средние/масштабы) учатся **только на train**. К test применяется `transform_features` с **теми же** числами. Почему? Если посчитать медиану по test — это подглядывание (утечка): в реальном закрытом тесте мы не должны ничего о нём знать заранее.

In [ ]:
# Готовим тест ТАК ЖЕ, как train: сначала новые признаки, потом матрица в ТОМ ЖЕ порядке колонок.
test_enriched = generate_features(test)  # те же 9 признаков для теста
test_raw = numeric_matrix(test_enriched, feature_names)  # тот же порядок колонок feature_names!
# Применяем TRAIN-параметры (medians/means/scales посчитаны на train выше) — fit здесь НЕТ.
test_transformed = transform_features(test_raw, medians, means, scales)
print("Тест обработан, размер:", test_transformed.shape)
# Средние на тесте НЕ ровно 0 (и это нормально!): параметры-то с train, а данные другие.
print("Средние первых 5 колонок теста:", test_transformed[:, :5].mean(axis=0).round(3))

## 6. Текстовые колонки: заполняем модами

**Мода** — самое частое значение. Пропуски в тексте заполняем модой с train. Наша финальная модель текст не использует, но механизм сохранён (и пригодится будущим моделям).

In [ ]:
# Учим моды на train: для каждой текстовой колонки — самое частое значение.
modes = fit_categorical_modes(train, CATEGORICAL_COLUMNS)
print("Моды:", modes)
# Демонстрация: испортим копию (вставим пропуски) и починим её модами.
demo = train[["region", "employment"]].copy()  # копия двух колонок
print("Пропусков до:", int(demo.isna().sum().sum()))  # их 0 — данные полные
# Искусственно ломаем первые 3 строки колонки region (ставим None = пропуск).
demo.loc[0:2, "region"] = None
print("Пропусков после поломки:", int(demo.isna().sum().sum()))
# Чиним: impute_categorical вернёт НОВУЮ таблицу (оригинал demo не меняется!).
fixed = impute_categorical(demo, modes)
print("Пропусков после починки:", int(fixed.isna().sum().sum()))
print(fixed.head(4))  # первые строки: пропуски заменились модой.

## Выводы ноутбука 03

1. `generate_features` добавляет 9 осмысленных признаков (экспозиции, отношения, возраст). Итого 61 числовой признак.
2. Пропуски заполняем **медианой** (устойчива к выбросам).
3. Масштаб выравниваем **стандартизацией**: (x − среднее) / разброс.
4. Параметры учатся **только на train** (`fit`), к test **применяются** (`transform`). Это закон, иначе утечка.
5. Текст чинится **модами** (самыми частыми значениями).

Дальше — **04_train_catboost**: как на этих обработанных числах обучается модель.